# Netball Betting Model — Exploratory Data Analysis

Analyzing SSN match data (2017-2025) to validate data quality, understand distributions, and check model assumptions before training.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Load data from SQLite
DB_PATH = Path("../data/netball.db")
conn = sqlite3.connect(DB_PATH)

matches = pd.read_sql("SELECT * FROM matches ORDER BY date", conn)
player_stats = pd.read_sql("SELECT * FROM player_stats", conn)
conn.close()

print(f"Matches: {len(matches)}")
print(f"Player stat rows: {len(player_stats)}")
print(f"Seasons: {sorted(matches['season'].unique())}")

Matches: 539
Player stat rows: 11041
Seasons: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


## 1. Data Quality

In [2]:
season_counts = matches.groupby("season").size().reset_index(name="match_count")
print(season_counts.to_string(index=False))
print(f"\nTotal matches: {len(matches)}")

 season  match_count
   2017           60
   2018           60
   2019           60
   2020           60
   2021           59
   2022           60
   2023           60
   2024           60
   2025           60

Total matches: 539


In [3]:
print("=== Missing values in matches ===")
missing = matches.isnull().sum()
print(missing[missing > 0] if missing.any() else "No missing values")

print("\n=== Score ranges ===")
print(f"Home scores: {matches['home_score'].min()} - {matches['home_score'].max()}")
print(f"Away scores: {matches['away_score'].min()} - {matches['away_score'].max()}")

# Check for zero-score matches (might indicate incomplete data)
zero_scores = matches[(matches["home_score"] == 0) | (matches["away_score"] == 0)]
if len(zero_scores) > 0:
    print(f"\nWARNING: {len(zero_scores)} matches with zero scores:")
    print(zero_scores[["match_id", "date", "home_team", "away_team", "home_score", "away_score"]])
else:
    print("\nNo zero-score matches \u2014 good.")

=== Missing values in matches ===
No missing values

=== Score ranges ===
Home scores: 22 - 86
Away scores: 25 - 82

No zero-score matches — good.


In [4]:
print("=== Teams per season ===")
for season in sorted(matches["season"].unique()):
    season_matches = matches[matches["season"] == season]
    teams = sorted(set(season_matches["home_team"].unique()) | set(season_matches["away_team"].unique()))
    print(f"\n{season} ({len(teams)} teams): {', '.join(teams)}")

=== Teams per season ===

2017 (8 teams): Adelaide Thunderbirds, Collingwood Magpies, GIANTS Netball, Melbourne Vixens, NSW Swifts, Queensland Firebirds, Sunshine Coast Lightning, West Coast Fever

2018 (8 teams): Adelaide Thunderbirds, Collingwood Magpies, GIANTS Netball, Melbourne Vixens, NSW Swifts, Queensland Firebirds, Sunshine Coast Lightning, West Coast Fever

2019 (8 teams): Adelaide Thunderbirds, Collingwood Magpies, GIANTS Netball, Melbourne Vixens, NSW Swifts, Queensland Firebirds, Sunshine Coast Lightning, West Coast Fever

2020 (8 teams): Adelaide Thunderbirds, Collingwood Magpies, GIANTS Netball, Melbourne Vixens, NSW Swifts, Queensland Firebirds, Sunshine Coast Lightning, West Coast Fever

2021 (8 teams): Adelaide Thunderbirds, Collingwood Magpies, GIANTS Netball, Melbourne Vixens, NSW Swifts, Queensland Firebirds, Sunshine Coast Lightning, West Coast Fever

2022 (8 teams): Adelaide Thunderbirds, Collingwood Magpies, GIANTS Netball, Melbourne Vixens, NSW Swifts, Queensla

In [5]:
print("=== Date ordering check ===")
for season in sorted(matches["season"].unique()):
    season_matches = matches[matches["season"] == season].sort_values("round_num")
    dates = pd.to_datetime(season_matches["date"].str[:10], errors="coerce")
    out_of_order = (dates.diff().dt.days < 0).sum()
    if out_of_order > 0:
        print(f"WARNING: {season} has {out_of_order} dates out of order")
    else:
        print(f"{season}: dates in order \u2713")

=== Date ordering check ===
